# 使用函数式 API
功能API允许您将 LangGraph 的主要功能（[持久性](https://langchain-ai.github.io/langgraph/concepts/persistence/)、[内存](https://langchain-ai.github.io/langgraph/how-tos/memory/add-memory/)、[人在环](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/)和[流式传输](https://langchain-ai.github.io/langgraph/concepts/streaming/)）添加到您的应用程序中，而只需对现有代码进行最少的更改。

> 有关功能 API 的概念信息，请参阅[功能 API](https://langchain-ai.github.io/langgraph/concepts/functional_api/)。


## 创建简单的工作流程¶
定义`entrypoint`时，输入仅限于函数的第一个参数。要传递多个输入，可以使用字典。

In [ ]:
@entrypoint(checkpointer=checkpointer)
def my_workflow(inputs: dict) -> int:
    value = inputs["value"]
    another_value = inputs["another_value"]
    ...

my_workflow.invoke({"value": 1, "another_value": 2})

### 简单的工作流程

In [ ]:
import uuid
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

# Task that checks if a number is even
@task
def is_even(number: int) -> bool:
    return number % 2 == 0

# Task that formats a message
@task
def format_message(is_even: bool) -> str:
    return "The number is even." if is_even else "The number is odd."

# Create a checkpointer for persistence
checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def workflow(inputs: dict) -> str:
    """Simple workflow to classify a number."""
    even = is_even(inputs["number"]).result()
    return format_message(even).result()

# Run the workflow with a unique thread ID
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = workflow.invoke({"number": 7}, config=config)
print(result)

使用大模型 (LLM) 撰写论文

此示例演示了如何在语法上使用@task和@entrypoint装饰器。由于提供了检查点，因此工作流结果将持久保存在检查点中。

In [ ]:
import uuid
from langchain.chat_models import init_chat_model
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

llm = init_chat_model('openai:gpt-3.5-turbo')

# Task: generate essay using an LLM
@task
def compose_essay(topic: str) -> str:
    """Generate an essay about the given topic."""
    return llm.invoke([
        {"role": "system", "content": "You are a helpful assistant that writes essays."},
        {"role": "user", "content": f"Write an essay about {topic}."}
    ]).content

# Create a checkpointer for persistence
checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def workflow(topic: str) -> str:
    """Simple workflow that generates an essay with an LLM."""
    return compose_essay(topic).result()

# Execute the workflow
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = workflow.invoke("the history of flight", config=config)
print(result)

## 并行执行
可以通过并发调用并等待结果来并行执行任务。这对于提升 IO 密集型任务（例如，调用 LLM 的 API）的性能非常有用。

In [ ]:
@task
def add_one(number: int) -> int:
    return number + 1

@entrypoint(checkpointer=checkpointer)
def graph(numbers: list[int]) -> list[str]:
    futures = [add_one(i) for i in numbers]
    return [f.result() for f in futures]

此示例演示如何使用 并行运行多个 LLM 调用@task。每次调用都会生成一个关于不同主题的段落，并将结果合并为单个文本输出。

In [ ]:
import uuid
from langchain.chat_models import init_chat_model
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

# Initialize the LLM model
llm = init_chat_model("openai:gpt-3.5-turbo")

# Task that generates a paragraph about a given topic
@task
def generate_paragraph(topic: str) -> str:
    response = llm.invoke([
        {"role": "system", "content": "You are a helpful assistant that writes educational paragraphs."},
        {"role": "user", "content": f"Write a paragraph about {topic}."}
    ])
    return response.content

# Create a checkpointer for persistence
checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def workflow(topics: list[str]) -> str:
    """Generates multiple paragraphs in parallel and combines them."""
    futures = [generate_paragraph(topic) for topic in topics]
    paragraphs = [f.result() for f in futures]
    return "\n\n".join(paragraphs)

# Run the workflow
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = workflow.invoke(["quantum computing", "climate change", "history of aviation"], config=config)
print(result)

## 调用图¶
由于功能API和图形 API共享相同的底层运行时，因此可以在同一个应用程序中一起使用。

API 参考：[入口点](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.entrypoint)| [StateGraph](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.state.StateGraph)


In [ ]:
from langgraph.func import entrypoint
from langgraph.graph import StateGraph

builder = StateGraph()
...
some_graph = builder.compile()

@entrypoint()
def some_workflow(some_input: dict) -> int:
    # Call a graph defined using the graph API
    result_1 = some_graph.invoke(...)
    # Call another graph defined using the graph API
    result_2 = another_graph.invoke(...)
    return {
        "result_1": result_1,
        "result_2": result_2
    }

## 调用其他入口点¶
您可以从入口点或任务中调用其他入口点。

In [ ]:
@entrypoint() # Will automatically use the checkpointer from the parent entrypoint
def some_other_workflow(inputs: dict) -> int:
    return inputs["value"]

@entrypoint(checkpointer=checkpointer)
def my_workflow(inputs: dict) -> int:
    value = some_other_workflow.invoke({"value": 1})
    return value

In [ ]:
import uuid
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import InMemorySaver

# Initialize a checkpointer
checkpointer = InMemorySaver()

# A reusable sub-workflow that multiplies a number
@entrypoint()
def multiply(inputs: dict) -> int:
    return inputs["a"] * inputs["b"]

# Main workflow that invokes the sub-workflow
@entrypoint(checkpointer=checkpointer)
def main(inputs: dict) -> dict:
    result = multiply.invoke({"a": inputs["x"], "b": inputs["y"]})
    return {"product": result}

# Execute the main workflow
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
print(main.invoke({"x": 6, "y": 7}, config=config))  # Output: {'product': 42}

## 流媒体¶
函数式 API使用与图谱 API相同的流式传输机制。请阅读[流式传输](https://langchain-ai.github.io/langgraph/concepts/streaming/)指南部分了解更多详情。

使用流式 API 来传输更新和自定义数据的示例。

API 参考：[entrypoint](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.entrypoint) | [InMemorySaver](https://langchain-ai.github.io/langgraph/reference/checkpoints/#langgraph.checkpoint.memory.InMemorySaver) | [get_stream_writer](https://langchain-ai.github.io/langgraph/reference/config/#langgraph.config.get_stream_writer)

In [1]:
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer 

checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def main(inputs: dict) -> int:
    writer = get_stream_writer() 
    writer("Started processing") 
    result = inputs["x"] * 2
    writer(f"Result is {result}") 
    return result

config = {"configurable": {"thread_id": "abc"}}

for mode, chunk in main.stream( 
    {"x": 5},
    stream_mode=["custom", "updates"], 
    config=config
):
    print(f"{mode}: {chunk}")

custom: Started processing
custom: Result is 10
updates: {'main': 10}


如果使用 Python 3.11 以下版本编写异步代码，get_stream_writer()则使用 将会无效。请StreamWriter直接使用类。更多详情，请参阅Python 3.11 以下版本的异步操作。

```python
from langgraph.types import StreamWriter

@entrypoint(checkpointer=checkpointer)
async def main(inputs: dict, writer: StreamWriter) -> int:
```

## 重试策略¶
API 参考：[InMemorySaver](https://langchain-ai.github.io/langgraph/reference/checkpoints/#langgraph.checkpoint.memory.InMemorySaver) | [entrypoint](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.entrypoint) | [task](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.task) | [RetryPolicy](https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.RetryPolicy)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.types import RetryPolicy

# This variable is just used for demonstration purposes to simulate a network failure.
# It's not something you will have in your actual code.
attempts = 0

# Let's configure the RetryPolicy to retry on ValueError.
# The default RetryPolicy is optimized for retrying specific network errors.
retry_policy = RetryPolicy(retry_on=ValueError)

@task(retry_policy=retry_policy) 
def get_info():
    global attempts
    attempts += 1

    if attempts < 2:
        raise ValueError('Failure')
    return "OK"

checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def main(inputs, writer):
    return get_info().result()

config = {
    "configurable": {
        "thread_id": "1"
    }
}

main.invoke({'any_input': 'foobar'}, config=config)

## 缓存任务¶

API 参考：[入口点](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.entrypoint)|[任务](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.task)

In [ ]:
import time
from langgraph.cache.memory import InMemoryCache
from langgraph.func import entrypoint, task
from langgraph.types import CachePolicy


@task(cache_policy=CachePolicy(ttl=120))  
def slow_add(x: int) -> int:
    time.sleep(1)
    return x * 2


@entrypoint(cache=InMemoryCache())
def main(inputs: dict) -> dict[str, int]:
    result1 = slow_add(inputs["x"]).result()
    result2 = slow_add(inputs["x"]).result()
    return {"result1": result1, "result2": result2}


for chunk in main.stream({"x": 5}, stream_mode="updates"):
    print(chunk)

#> {'slow_add': 10}
#> {'slow_add': 10, '__metadata__': {'cached': True}}
#> {'main': {'result1': 10, 'result2': 10}}

## 发生错误后恢复¶
API 参考：[InMemorySaver](https://langchain-ai.github.io/langgraph/reference/checkpoints/#langgraph.checkpoint.memory.InMemorySaver) |[入口点](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.entrypoint)|[任务](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.task)| [StreamWriter](https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.StreamWriter)

In [ ]:
import time
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.types import StreamWriter

# This variable is just used for demonstration purposes to simulate a network failure.
# It's not something you will have in your actual code.
attempts = 0

@task()
def get_info():
    """
    Simulates a task that fails once before succeeding.
    Raises an exception on the first attempt, then returns "OK" on subsequent tries.
    """
    global attempts
    attempts += 1

    if attempts < 2:
        raise ValueError("Failure")  # Simulate a failure on the first attempt
    return "OK"

# Initialize an in-memory checkpointer for persistence
checkpointer = InMemorySaver()

@task
def slow_task():
    """
    Simulates a slow-running task by introducing a 1-second delay.
    """
    time.sleep(1)
    return "Ran slow task."

@entrypoint(checkpointer=checkpointer)
def main(inputs, writer: StreamWriter):
    """
    Main workflow function that runs the slow_task and get_info tasks sequentially.

    Parameters:
    - inputs: Dictionary containing workflow input values.
    - writer: StreamWriter for streaming custom data.

    The workflow first executes `slow_task` and then attempts to execute `get_info`,
    which will fail on the first invocation.
    """
    slow_task_result = slow_task().result()  # Blocking call to slow_task
    get_info().result()  # Exception will be raised here on the first attempt
    return slow_task_result

# Workflow execution configuration with a unique thread identifier
config = {
    "configurable": {
        "thread_id": "1"  # Unique identifier to track workflow execution
    }
}

# This invocation will take ~1 second due to the slow_task execution
try:
    # First invocation will raise an exception due to the `get_info` task failing
    main.invoke({'any_input': 'foobar'}, config=config)
except ValueError:
    pass  # Handle the failure gracefully

当我们恢复执行时，我们不需要重新运行，slow_task因为其结果已经保存在检查点中。


```shell
main.invoke(None, config=config)

'Ran slow task.'
```

## 人机交互¶

函数式 API 支持使用`interrupt`函数和 `Command `原语的人机交互工作流。

基本的人机循环工作流程¶
我们将创建三个任务：

- 追加"bar"。
- 暂停以等待人工输入。恢复时，附加人工输入。
- 追加"qux"。

API 参考：[入口点](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.entrypoint)|[任务](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.task)|[命令](https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.Command)|[中断](https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.interrupt)

In [ ]:
from langgraph.func import entrypoint, task
from langgraph.types import Command, interrupt


@task
def step_1(input_query):
    """Append bar."""
    return f"{input_query} bar"


@task
def human_feedback(input_query):
    """Append user input."""
    feedback = interrupt(f"Please provide feedback: {input_query}")
    return f"{input_query} {feedback}"


@task
def step_3(input_query):
    """Append qux."""
    return f"{input_query} qux"

我们现在可以在入口点中组合这些任务：

API 参考：InMemorySaver

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()


@entrypoint(checkpointer=checkpointer)
def graph(input_query):
    result_1 = step_1(input_query).result()
    result_2 = human_feedback(result_1).result()
    result_3 = step_3(result_2).result()

    return result_3

interrupt（） 在任务中调用，使人类能够查看和编辑上一个任务的输出。先前任务的结果（在本例中为 step_1）将持久化，因此它们不会在interrupt后再次运行。

In [ ]:
config = {"configurable": {"thread_id": "1"}}

for event in graph.stream("foo", config):
    print(event)
    print("\n")

请注意，我们在step_1后暂停了interrupt。interrupt提供恢复运行的指令。为了恢复，我们发出一个命令，其中包含human_feedback任务所需的数据。

In [ ]:
# Continue execution
for event in graph.stream(Command(resume="baz"), config):
    print(event)
    print("\n")

恢复后，运行将继续执行剩余步骤并按预期终止。

### 为了在执行之前检查工具调用，我们添加了一个review_tool_call调用 的函数interrupt。调用此函数时，执行将暂停，直到我们发出命令恢复执行。

收到工具调用后，我们的函数将interrupt进行人工审核。此时，我们可以选择以下方式：

- 接受工具调用
- 修改工具调用并继续
- 生成自定义工具消息（例如，指示模型重新格式化其工具调用）

In [ ]:
from typing import Union

def review_tool_call(tool_call: ToolCall) -> Union[ToolCall, ToolMessage]:
    """Review a tool call, returning a validated version."""
    human_review = interrupt(
        {
            "question": "Is this correct?",
            "tool_call": tool_call,
        }
    )
    review_action = human_review["action"]
    review_data = human_review.get("data")
    if review_action == "continue":
        return tool_call
    elif review_action == "update":
        updated_tool_call = {**tool_call, **{"args": review_data}}
        return updated_tool_call
    elif review_action == "feedback":
        return ToolMessage(
            content=review_data, name=tool_call["name"], tool_call_id=tool_call["id"]
        )

我们现在可以更新`entrypoint`以查看生成的工具调用。如果工具调用被接受或修改，我们将以与以前相同的方式执行。否则，我们只需附加人类提供的 ToolMessage。先前任务的结果（在本例中为初始模型调用）将持久化，因此它们不会在`interrupt`后再次运行。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt


checkpointer = InMemorySaver()


@entrypoint(checkpointer=checkpointer)
def agent(messages, previous):
    if previous is not None:
        messages = add_messages(previous, messages)

    llm_response = call_model(messages).result()
    while True:
        if not llm_response.tool_calls:
            break

        # Review tool calls
        tool_results = []
        tool_calls = []
        for i, tool_call in enumerate(llm_response.tool_calls):
            review = review_tool_call(tool_call)
            if isinstance(review, ToolMessage):
                tool_results.append(review)
            else:  # is a validated tool call
                tool_calls.append(review)
                if review != tool_call:
                    llm_response.tool_calls[i] = review  # update message

        # Execute remaining tool calls
        tool_result_futures = [call_tool(tool_call) for tool_call in tool_calls]
        remaining_tool_results = [fut.result() for fut in tool_result_futures]

        # Append to message list
        messages = add_messages(
            messages,
            [llm_response, *tool_results, *remaining_tool_results],
        )

        # Call model again
        llm_response = call_model(messages).result()

    # Generate final response
    messages = add_messages(messages, llm_response)
    return entrypoint.final(value=llm_response, save=messages)

## 短期记忆¶
短期记忆允许存储同一线程 ID在不同调用之间存储的信息。更多详情，请参阅短期记忆。

### 管理检查点¶
您可以查看和删除检查点存储的信息。

查看线程状态（检查点）¶




In [ ]:
config = {
    "configurable": {
        "thread_id": "1",
        # optionally provide an ID for a specific checkpoint,
        # otherwise the latest checkpoint is shown
        # "checkpoint_id": "1f029ca3-1f5b-6704-8004-820c16b69a5a"

    }
}
graph.get_state(config)

```shell
StateSnapshot(
    values={'messages': [HumanMessage(content="hi! I'm bob"), AIMessage(content='Hi Bob! How are you doing today?), HumanMessage(content="what's my name?"), AIMessage(content='Your name is Bob.')]}, next=(), 
    config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f029ca3-1f5b-6704-8004-820c16b69a5a'}},
    metadata={
        'source': 'loop',
        'writes': {'call_model': {'messages': AIMessage(content='Your name is Bob.')}},
        'step': 4,
        'parents': {},
        'thread_id': '1'
    },
    created_at='2025-05-05T16:01:24.680462+00:00',
    parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f029ca3-1790-6b0a-8003-baf965b6a38f'}}, 
    tasks=(),
    interrupts=()
)
```

### 查看线程的历史记录（检查点）¶

In [ ]:
config = {
    "configurable": {
        "thread_id": "1"
    }
}
list(graph.get_state_history(config))

#### 将返回值与保存的值分离¶
用于entrypoint.final将返回给调用者的内容与检查点中持久保存的内容分离。这在以下情况下很有用：

- 您想要返回计算结果（例如摘要或状态），但保存不同的内部值以供下次调用使用。
- 您需要控制下次运行时传递给上一个参数的内容。

API 参考：入口点| InMemorySaver

In [ ]:
from typing import Optional
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def accumulate(n: int, *, previous: Optional[int]) -> entrypoint.final[int, int]:
    previous = previous or 0
    total = previous + n
    # Return the *previous* value to the caller but save the *new* total to the checkpoint.
    return entrypoint.final(value=previous, save=total)

config = {"configurable": {"thread_id": "my-thread"}}

print(accumulate.invoke(1, config=config))  # 0
print(accumulate.invoke(2, config=config))  # 1
print(accumulate.invoke(3, config=config))  # 3

### 聊天机器人示例¶
这是一个使用函数式 API 和检查点的简单聊天机器人示例InMemorySaver。该机器人能够记住之前的对话，并从上次中断的地方继续。

API 参考：BaseMessage | add_messages | entrypoint | task | InMemorySaver | ChatAnthropic

In [ ]:
from langchain_core.messages import BaseMessage
from langgraph.graph import add_messages
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-3-5-sonnet-latest")

@task
def call_model(messages: list[BaseMessage]):
    response = model.invoke(messages)
    return response

checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def workflow(inputs: list[BaseMessage], *, previous: list[BaseMessage]):
    if previous:
        inputs = add_messages(previous, inputs)

    response = call_model(inputs).result()
    return entrypoint.final(value=response, save=add_messages(inputs, response))

config = {"configurable": {"thread_id": "1"}}
input_message = {"role": "user", "content": "hi! I'm bob"}
for chunk in workflow.stream([input_message], config, stream_mode="values"):
    chunk.pretty_print()

input_message = {"role": "user", "content": "what's my name?"}
for chunk in workflow.stream([input_message], config, stream_mode="values"):
    chunk.pretty_print()

## 长期记忆
长期记忆允许跨不同的线程 ID存储信息。这对于在一次对话中学习特定用户的信息，并在另一个对话中使用它非常有用。

### 工作流程¶
工作流和代理指南提供了有关如何使用功能 API 构建工作流的更多示例。
### 代理商¶
- 如何从头开始创建代理（功能 API）：展示如何使用功能 API 从头开始创建一个简单的代理。
- 如何构建多代理网络：展示如何使用功能 API 构建多代理网络。
- 如何在多代理应用程序中添加多轮对话（功能性 API）：允许最终用户与一个或多个代理进行多轮对话。
### 与其他库集成¶
使用功能 API 将 LangGraph 的功能添加到其他框架：将持久性、内存和流式传输等 LangGraph 功能添加到其他未提供开箱即用的代理框架。